# 📘 Module 5.2 — GPT Architecture

**Goal:** Understand the GPT (Generative Pre-trained Transformer) architecture — the foundation of modern LLMs.

## GPT = Decoder-Only Transformer

GPT uses only the **decoder** part of the Transformer with **causal (masked) attention** — each token can only attend to previous tokens.

```
Encoder-Decoder: BERT (understands), T5 (translates)
Decoder-Only:    GPT (generates) ← This is what we build
```

### ADAS Relevance
- Scene narration: "A pedestrian is crossing from the left"
- Driving policy generation
- Code generation for perception pipelines

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

## 1. Causal (Masked) Self-Attention

The key difference from regular attention: each position can only attend to **earlier positions** (and itself).

```
Regular attention:    Token 3 sees [1, 2, 3, 4, 5]  (all)
Causal attention:     Token 3 sees [1, 2, 3, -, -]  (only past + self)
```

This enables **autoregressive generation** — predicting one token at a time.

In [ ]:
# --- Causal Mask ---
seq_len = 8
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full attention (encoder)
axes[0].imshow(torch.zeros(seq_len, seq_len), cmap='Greens', vmin=0, vmax=1)
axes[0].set_title('Full Attention (Encoder)\nEvery token sees everything')

# Causal attention (decoder/GPT)
axes[1].imshow(causal_mask.float(), cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Causal Attention (GPT)\nRed = masked (cannot attend)')

for ax in axes:
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.tight_layout()
plt.show()

## 2. GPT Block

In [ ]:
class GPTBlock(nn.Module):
    """A single GPT transformer block."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        seq_len = x.size(1)
        # Causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        
        # Pre-norm architecture (GPT-2 style)
        # Self-attention with causal mask
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + self.dropout(attn_out)  # Residual
        
        # Feed-forward
        x = x + self.ffn(self.ln2(x))   # Residual
        
        return x

block = GPTBlock(d_model=256, num_heads=8, d_ff=1024)
x = torch.randn(2, 20, 256)
print(f"GPT Block: {x.shape} → {block(x).shape}")

In [ ]:
class MiniGPT(nn.Module):
    """
    Minimal GPT implementation.
    Demonstrates the complete architecture.
    """
    
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=6,
                 d_ff=1024, max_seq_len=512, dropout=0.1):
        super().__init__()
        
        # Token + Position embeddings
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Output
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying (embedding weights = output weights)
        self.head.weight = self.token_embed.weight
    
    def forward(self, idx):
        B, T = idx.shape
        
        # Embeddings
        positions = torch.arange(T, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        x = self.dropout(x)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Output logits
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        return logits
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive text generation."""
        for _ in range(max_new_tokens):
            # Get logits for last position
            logits = self(idx)[:, -1, :] / temperature
            
            # Sample from probability distribution
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            idx = torch.cat([idx, next_token], dim=1)
        
        return idx

# Create model
model = MiniGPT(vocab_size=10000, d_model=256, num_heads=8, num_layers=4)

print(f"MiniGPT Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nFor reference:")
print(f"  GPT-2 Small:  124M parameters")
print(f"  GPT-2 Medium: 355M parameters")
print(f"  GPT-3:        175B parameters")
print(f"  GPT-4:        ~1.8T parameters (estimated)")
print(f"  LLaMA-3 8B:   8B parameters")

In [ ]:
# --- Test generation (with random weights = random output) ---
model.eval()
prompt = torch.tensor([[1, 2, 3, 4, 5]])  # Start tokens
generated = model.generate(prompt, max_new_tokens=10, temperature=0.8)

print(f"Prompt tokens:    {prompt[0].tolist()}")
print(f"Generated tokens: {generated[0].tolist()}")
print("\n💡 With random weights, output is random.")
print("After training on text data, it generates coherent language!")

## 3. Training Objective: Next Token Prediction

GPT is trained to predict the next token given all previous tokens:

```
Input:  "The car is approaching the"
Target: "car is approaching the intersection"

At each position, predict the NEXT token.
Loss = CrossEntropy between predicted and actual next token.
```

In [ ]:
# --- Next Token Prediction Training ---

# Simulated training data
batch_size = 4
seq_len = 32
data = torch.randint(0, 10000, (batch_size, seq_len + 1))

# Input: all tokens except last
# Target: all tokens except first (shifted by 1)
inputs = data[:, :-1]   # (batch, seq_len)
targets = data[:, 1:]   # (batch, seq_len) — shifted!

# Forward pass
logits = model(inputs)   # (batch, seq_len, vocab_size)

# Loss: cross-entropy between predictions and targets
loss = F.cross_entropy(
    logits.view(-1, 10000),  # Flatten: (batch*seq_len, vocab_size)
    targets.view(-1)          # Flatten: (batch*seq_len,)
)

print(f"Input shape:  {inputs.shape}")
print(f"Target shape: {targets.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"Random baseline loss: {np.log(10000):.4f} (ln(vocab_size))")

---
## ✅ Key Takeaways

1. **GPT** is a decoder-only Transformer using causal attention
2. **Causal mask** prevents tokens from attending to future positions
3. **Training:** Predict the next token (self-supervised — no labels needed!)
4. **Generation:** Autoregressive — predict one token at a time
5. **Weight tying** (embed = output weights) reduces parameters
6. **Scaling** model size, data, and compute → better performance (scaling laws)

---
## 📖 Next Steps
➡️ **Next notebook:** [03_finetuning_prompt_engineering.ipynb](03_finetuning_prompt_engineering.ipynb) — Fine-tuning and prompt engineering